# Lookahead check and walk-forward splits (Week 5)

**Owner:** Salah · **With:** Fatima (Evaluation / backtesting) · **Code:** `src/walk_forward.py` · **Checklist:** `docs/lookahead_checklist.md`

Week 5 tasks: (1) walk-forward train/test splits, (2) confirm no feature uses future data, (3) a lookahead-bias checklist for the team.

**What "lookahead" means here:** a model that runs at the close of day *t* may only use information published by the close of day *t*. Anything else makes backtest results look better than a live system could achieve.

Sections:
1. Truncation test: recompute each feature with all data after day *t* deleted; the value at *t* must not change
2. Data that is published late (weekly FRED indexes)
3. The stress labels and the old `stress` column in `daily_panel`
4. Data cleaning and sentiment
5. Audit of the Week 5 model notebooks
6. Walk-forward splits, and what changes when the leaks are fixed
7. Findings

**Before running:** the shared `data/processed/` folder from the team Drive. Runtime is under a minute.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src import walk_forward as wf, windows as wn, returns as rt, data_loading as dl
from src import clustering as cl, stress_labels as sl

PROC = ROOT / "data" / "processed"
TOL = 1e-9
rets = rt.load_returns()
panel = pd.read_parquet(PROC / "daily_panel.parquet")
labels = pd.read_parquet(PROC / "stress_labels_daily.parquet")

## 1. Truncation test

For each feature, pick dates just before six stress episodes, delete all data after that date, recompute, and compare with the value in the shared feature file. Any difference above rounding error (1e-9) means the feature used future data.

In [2]:
conn = pd.read_parquet(PROC / "features" / "connectedness_60d.parquet")
clus = pd.read_parquet(PROC / "features" / "clusters_market_adjusted_20d.parquet")
try:
    from src import centrality as ce
    cent = pd.read_parquet(PROC / "features" / "centrality_60d.parquet")
except (ImportError, FileNotFoundError):
    ce = cent = None
    print("centrality not available (merge week4-centrality and add the feature file); skipping")

CHECK_DATES = ["2007-07-25", "2008-09-10", "2011-07-20", "2020-02-12", "2022-02-16", "2025-03-12"]
sector_rets = panel[[f"{t}_ret" for t in dl.SECTOR_ETFS]]
rows = []
for d in CHECK_DATES:
    t = conn.index[conn.index <= d][-1]
    cut = wn.connectedness(rets.loc[:t], start=t, end=t)
    cols = ["avg_corr", "pc1_share", "absorption"]
    rows.append(("features/connectedness_60d", t, (cut.loc[t, cols] - conn.loc[t, cols]).abs().max()))

    a = dl.rolling_avg_correlation(sector_rets.loc[:t], 60).iloc[-1]
    b = dl.absorption_ratio(sector_rets.loc[:t], 250).iloc[-1]
    rows.append(("daily_panel avg_corr_60d, absorption_250d", t,
                 max(abs(a - panel.loc[t, "avg_corr_60d"]), abs(b - panel.loc[t, "absorption_250d"]))))

    tc = clus.index[clus.index <= d][-1]
    prev = clus.index[clus.index < tc][-1]          # one earlier window so stability can be checked
    f, _ = cl.run_over_time(rets.loc[:tc], cl.RECOMMENDED, window=60, step=20, start=prev, end=tc)
    cols = ["n_edges", "n_communities", "modularity", "largest_share", "stability"]
    rows.append(("features/clusters_market_adjusted_20d", tc, (f.loc[tc, cols] - clus.loc[tc, cols]).abs().max()))

    if ce is not None:
        tz = cent.index[cent.index <= d][-1]
        f, _ = ce.centrality_over_time(rets.loc[:tz], start=tz, end=tz)
        cols = ["strength_gini", "strength_top10", "degree_central", "evc_top10", "raw_evc_eff_share"]
        rows.append(("features/centrality_60d", tz, (f.loc[tz, cols] - cent.loc[tz, cols]).abs().max()))

trunc = pd.DataFrame(rows, columns=["feature", "date", "max_abs_diff"])
trunc["result"] = np.where(trunc["max_abs_diff"] < TOL, "PASS", "FAIL")
display(trunc.groupby("feature").agg(dates_checked=("date", "count"), worst_diff=("max_abs_diff", "max"),
                                     result=("result", lambda r: "PASS" if (r == "PASS").all() else "FAIL")))

centrality not available (merge week4-centrality and add the feature file); skipping


,dates_checked,worst_diff,result
feature,,,
"daily_panel avg_corr_60d, absorption_250d",6,1.110223e-16,PASS
features/clusters_market_adjusted_20d,6,1.110223e-15,PASS
features/connectedness_60d,6,3.885781e-16,PASS


## 2. Data published late: weekly FRED stress indexes

`STLFSI4` and `NFCI` are dated the Friday that ends each week, but they are published days later: NFCI the following Wednesday ([Chicago Fed](https://www.chicagofed.org/research/data/nfci/current-data)), STLFSI4 the following Wednesday or Thursday ([FRED](https://fred.stlouisfed.org/series/STLFSI4)). `daily_panel.parquet` forward-fills them by **observation** date, so on Friday it already shows that Friday's value, about a week before anyone could see it. The data dictionary says to shift them; `wf.lag_by_release` does that (5 trading days).

In [3]:
for col in ["STLFSI4", "NFCI"]:
    changes = panel[col].dropna()
    changes = changes[changes.diff() != 0].index
    print(f"{col}: value changes on {changes.day_name().value_counts().idxmax()}s "
          f"({changes.day_name().value_counts(normalize=True).max():.0%} of changes)")

lagged = wf.lag_by_release(panel)
d = pd.Timestamp("2008-10-10")
print(f"\nExample, {d.date()} (Friday): STLFSI4 in panel = {panel.loc[d, 'STLFSI4']:.2f}, "
      f"value actually published by then = {lagged.loc[d, 'STLFSI4']:.2f}")

STLFSI4: value changes on Fridays (97% of changes)
NFCI: value changes on Fridays (97% of changes)

Example, 2008-10-10 (Friday): STLFSI4 in panel = 9.67, value actually published by then = 7.30


A second, smaller issue: FRED serves the **latest revised** history. NFCI in particular is re-estimated every week, so old values change after the fact. A fully clean test would use the values as first published (FRED's ALFRED archive). We note this as a limitation rather than fix it, since VIX, our main benchmark, is never revised.

## 3. Stress labels, and the old `stress` column

Our labels use a **centered** volatility window, so the label on day *t* needs returns up to *t* + 10. That is fine for a target, never for a feature. The truncation test shows it: with data cut at *t*, the label at *t* cannot even be computed.

In [4]:
g = pd.read_parquet(PROC / "sp500_close.parquet")["^GSPC"].dropna()
for d in ["2008-09-02", "2020-02-20"]:
    t = g.index[g.index <= d][-1]
    cut = sl.daily_labels(g.loc[:t])
    print(f"{t.date()}: label with full data = {labels.loc[t, 'stress']:.0f}, "
          f"with data cut at that date = {cut.loc[t, 'stress']}")

2008-09-02: label with full data = 1, with data cut at that date = nan
2020-02-20: label with full data = 1, with data cut at that date = nan


`daily_panel.parquet` also has a column named **`stress`**. It is **not** our label: it is an early placeholder from `01_data_collection.ipynb` (STLFSI4 above 1, not shifted for its release lag). Using it as a feature leaks a week of STLFSI4; using it as the target evaluates against the wrong definition.

In [5]:
both = pd.DataFrame({"panel_stress": panel["stress"], "official": labels["stress"]}).dropna()
print(f"Agreement with the official labels: {(both.panel_stress == both.official).mean():.1%} of days")
print(f"Official stress days the panel column misses: "
      f"{((both.official == 1) & (both.panel_stress == 0)).sum()} of {(both.official == 1).sum()}")

Agreement with the official labels: 89.4% of days
Official stress days the panel column misses: 341 of 630


## 4. Data cleaning and sentiment

**Return cleaning** (`src/returns.py`) decides some data errors using a little future data: a spike is removed only if it reverses the **next** day, a stale price is one that stays unchanged for several days, and manual overrides were chosen by looking at the full history. In live use the bad tick would be visible for a day or two. It affects very few returns, so we accept it for research and list it as a limitation.

In [6]:
q = pd.read_csv(PROC / "returns_quality.csv")
print(q["reason"].value_counts().to_string())
removed = q["reason"].isin(["spike_removed", "stale_removed", "override_drop"]).sum()
print(f"\nRemoved member-day returns: {removed:,} of {rets.notna().sum().sum() + removed:,} "
      f"({removed / (rets.notna().sum().sum() + removed):.3%})")

reason
override_drop            2794
stale_removed            1770
large_move_kept            66
repeated_jumps_review      14
spike_removed              12
override_keep               8

Removed member-day returns: 4,576 of 2,124,438 (0.215%)


**Sentiment** (`notebooks/04_sentiment_full_corpus.ipynb`, checked by reading the code since `sec_sentiment_daily.parquet` is not in the shared Drive yet): each filing is mapped to the first trading day on or after its filing date and then shifted one more trading day. Filings accepted after the 4 p.m. close keep that day's date on EDGAR, so the extra day is needed, and it is there. **PASS.**

## 5. Audit of the Week 5 model notebooks

| Notebook | Check | Result |
|---|---|---|
| Grace `10_volatility_forecasting` | target uses returns after *t* only; features trailing; cluster features joined backward | PASS |
| Grace `10_volatility_forecasting` | split purges training rows whose 20-day target reaches the next split | PASS |
| Grace `11_walk_forward_forecasting` | training only on targets that ended before each forecast origin | PASS |
| Fatima `10_evaluation_backtesting` | `STLFSI4` and `NFCI` used without the release lag | **FAIL** (about one week of lookahead) |
| Fatima `10_evaluation_backtesting` | one 80/20 split with no purge; the last 10 training labels use returns from the test period | minor leak |
| Fatima `10_evaluation_backtesting` | target is stress on the **same day**, so it measures detection, not early warning (her notebook says so) | design issue for Week 6 |
| Fatima `11_robustness_checks` | same three issues: FRED not lagged, one 80/20 split, same-day target (the `center=True` volatility windows are used only as labels, which is fine) | **FAIL** (same fix) |

Section 6 measures how much these matter.

## 6. Walk-forward splits, and what changes when the leaks are fixed

`wf.walk_forward_splits` gives one fold per calendar year from 2010: train on everything before the year, test on the year, refit, move on. Training rows whose label reaches into the test year are dropped (purged).

We rerun Fatima's comparison (VIX only vs VIX + five indicators, same logistic regression) four ways:

* **A** her setup: same-day target, FRED not lagged, one 80/20 split
* **A2** same, with FRED lagged
* **B** same-day target, FRED lagged, walk-forward 2010-2026
* **C** early warning: "stress starts within the next 20 trading days", using only days not already in stress, FRED lagged, walk-forward

PR-AUC is the main score (stress days are rare); compare it with the base rate, which is what a random guess scores.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

FEATS = ["VIXCLS", "STLFSI4", "NFCI", "SPY_ret", "avg_corr_60d", "absorption_250d"]
MODELS = {"VIX only": ["VIXCLS"], "Combined": FEATS}

def new_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))

def scores(y, p):
    return {"ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
            "base rate": y.mean(), "test days": len(y)}

def single_split(df):
    k = int(len(df) * 0.8)
    tr, te = df.iloc[:k], df.iloc[k:]
    return {m: scores(te.y, new_model().fit(tr[c], tr.y).predict_proba(te[c])[:, 1]) for m, c in MODELS.items()}

def walk_forward(df, first_year=2010):
    folds = list(wf.walk_forward_splits(df.index, df["label_end"], first_test_year=first_year))
    wf.check_folds(folds, df["label_end"])
    preds = []
    for f in folds:
        tr, te = df.loc[f.train], df.loc[f.test]
        out = pd.DataFrame({"y": te.y}, index=te.index)
        for m, c in MODELS.items():
            out[m] = new_model().fit(tr[c], tr.y).predict_proba(te[c])[:, 1]
        preds.append(out)
    p = pd.concat(preds)
    return {m: scores(p.y, p[m]) for m in MODELS}, folds

raw, lag = panel[FEATS], wf.lag_by_release(panel[FEATS])
same_day, ahead20 = wf.stress_target(labels["stress"], 0), wf.stress_target(labels["stress"], 20)

data_A = raw.join(same_day, how="inner").dropna(subset=FEATS + ["y"])
data_B = lag.join(same_day, how="inner").dropna(subset=FEATS + ["y"])
data_C = lag.join(ahead20, how="inner").dropna(subset=FEATS + ["y"])
data_C = data_C[data_C["in_stress"] == 0]

results = {"A  Fatima setup": single_split(data_A), "A2 FRED lagged": single_split(data_B)}
results["B  walk-forward"], folds_B = walk_forward(data_B)
results["C  early warning, walk-forward"], folds_C = walk_forward(data_C)

table = pd.DataFrame({(s, m): v for s, r in results.items() for m, v in r.items()}).T
table["test days"] = table["test days"].astype(int)
display(table.round(3))

ROC-AUC  PR-AUC  base rate  test days
A  Fatima setup                VIX only    0.948   0.724      0.082       1094
                               Combined    0.958   0.751      0.082       1094
A2 FRED lagged                 VIX only    0.948   0.724      0.082       1094
                               Combined    0.951   0.721      0.082       1094
B  walk-forward                VIX only    0.943   0.761      0.098       4221
                               Combined    0.918   0.711      0.098       4221
C  early warning, walk-forward VIX only    0.571   0.068      0.053       3788
                               Combined    0.490   0.066      0.053       3788

In [8]:
print("Folds for setup C (one per test year):")
display(wf.fold_table(folds_C))

Folds for setup C (one per test year):


,test_year,train_start,train_end,test_start,test_end,n_train,n_test,purged
0,2010,2005-01-19,2009-11-17,2010-01-04,2010-12-31,1001,215,30
1,2011,2005-01-19,2010-11-17,2011-01-03,2011-12-30,1216,155,30
2,2012,2005-01-19,2011-07-22,2012-01-03,2012-12-31,1386,250,15
3,2013,2005-01-19,2012-11-15,2013-01-02,2013-12-31,1621,252,30
4,2014,2005-01-19,2013-11-15,2014-01-02,2014-12-31,1873,252,30
5,2015,2005-01-19,2014-11-17,2015-01-02,2015-12-31,2125,231,30
6,2016,2005-01-19,2015-11-17,2016-01-04,2016-12-30,2356,252,30
7,2017,2005-01-19,2016-11-16,2017-01-03,2017-12-29,2608,251,30
8,2018,2005-01-19,2017-11-15,2018-01-02,2018-12-10,2859,224,30
9,2019,2005-01-19,2018-11-23,2019-01-10,2019-12-31,3103,246,10


## 7. Findings

All numbers come from the outputs above.

1. **Every trailing feature passes.** Connectedness, the panel's `avg_corr_60d` and `absorption_250d`, the cluster features and the centrality features give identical values (differences under 2e-15) when all later data is deleted, at six dates each (Section 1).
2. **`STLFSI4` and `NFCI` leak about a week.** Both change on Fridays (97% of changes) but are published the next Wednesday or Thursday. On 2008-10-10 the panel shows STLFSI4 = 9.67 while the value actually published by then was 7.30 (Section 2). Fix: `wf.lag_by_release`.
3. **The leak created the combined model's edge.** With the lag fixed, the combined model's PR-AUC drops from 0.751 to 0.721, below VIX alone at 0.724 (setups A and A2). Over 2010-2026 walk-forward it is clearly worse: 0.711 vs 0.761 (setup B).
4. **Same-day detection is nearly circular.** Our labels are defined from realized volatility and VIX is the market's estimate of volatility, so a high same-day score (ROC-AUC about 0.95) mostly confirms that VIX is high when volatility is high.
5. **Real early warning is much harder.** Predicting stress within the next 20 trading days from calm days, both models are close to a coin flip: ROC-AUC 0.571 (VIX) and 0.490 (combined), PR-AUC 0.068 and 0.066 against a base rate of 0.053 (setup C). This is the honest starting point for Week 6, not a failure of the framework.
6. **Labels are targets only.** The centered labels cannot be computed on the day itself (Section 3). The old `stress` column in `daily_panel.parquet` is a different, leaky definition that agrees with the official labels on 89.4% of days but misses 341 of 630 stress days; nobody should use it.
7. **Accepted limitations.** Return cleaning removes 0.215% of member-day returns using a day or two of future data (Section 4), and FRED serves revised history, not first-published values (Section 2).

**Recommendation for Week 6:** use setup C as the team's test: `wf.stress_target(..., horizon=20)`, calm days only, `wf.walk_forward_splits` from 2010, FRED lagged. Fatima's metrics plug in unchanged. The horizon (20 days) should be agreed by the team.

In [9]:
out = ROOT / "notebooks" / "outputs" / "salah_week5"
out.mkdir(parents=True, exist_ok=True)
trunc.to_csv(out / "truncation_test.csv", index=False)
table.to_csv(out / "walk_forward_comparison.csv")
wf.fold_table(folds_C).to_csv(out / "walk_forward_folds.csv", index=False)
print("Saved to", out.relative_to(ROOT))

Saved to notebooks\outputs\salah_week5
